# **SMAP 1km : NSIDC-0779 v1.0**

In [ ]:
!pip install earthaccess rasterio shapely pyproj matplotlib numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/201.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 96.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.1.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.1.0 which is incompatible.


In [ ]:
import earthaccess
from pathlib import Path

In [ ]:
# ------------------------------------------------------------------
# 1. Authenticate with Earthdata
# ------------------------------------------------------------------
# This will prompt for username/password the FIRST time
auth = earthaccess.login(persist=True)

In [ ]:
# ------------------------------------------------------------------
# 2. Define dataset parameters
# ------------------------------------------------------------------
SHORT_NAME = "NSIDC-0779"   # Dataset short name
VERSION = "1"              # Dataset version

# Optional: temporal subset
temporal = ("2016-01-01", "2025-12-31")

# Optional: spatial subset (min_lon, min_lat, max_lon, max_lat)
# Remove this if you want global data
# bbox = (-180, -90, 180, 90)

# CV Extent
# Your bounding box (EPSG:4326 / WGS84)
bbox = (
    -122.62822473238745,  # min longitude
     34.91014552706419,   # min latitude
    -118.58047102622170,  # max longitude
     40.68211245220932    # max latitude
)


OUTPUT_DIR = Path("./NSIDC_0779_1")

OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ------------------------------------------------------------------
# 3. Search for granules
# ------------------------------------------------------------------
granules = earthaccess.search_data(
    short_name=SHORT_NAME,
    version=VERSION,
    temporal=temporal,
    bounding_box=bbox
)

n = len(granules)
print(f"Found {n} granules")

Found 2623 granules


In [ ]:
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import from_bounds
from shapely.geometry import box
from pathlib import Path
import tempfile
import os, math


BATCH_SIZE = 25
TARGET_CRS = "EPSG:4326"


# --------------------------------------------------
# PROCESS LOOP (FAST)
# --------------------------------------------------

num_batches = math.ceil(n / BATCH_SIZE)
for b in range(num_batches):

    print(f"\n🚀 Batch {b+1}/{num_batches}")

    batch_granules = granules[b*BATCH_SIZE:(b+1)*BATCH_SIZE]

    with tempfile.TemporaryDirectory() as batch_tmp:
        batch_tmp = Path(batch_tmp)

        # -------------------------
        # 1. Download batch
        # -------------------------
        raw_files = earthaccess.download(
            batch_granules,
            local_path=batch_tmp
        )

        print(f"  Downloaded {len(raw_files)} files")

        # -------------------------
        # 2. Process batch
        # -------------------------
        for raw_file in raw_files:

            raw_file = Path(raw_file)

            try:
                with rasterio.open(raw_file) as src:

                    with WarpedVRT(
                        src,
                        crs=TARGET_CRS,
                        resampling=rasterio.enums.Resampling.nearest
                    ) as vrt:

                        window = from_bounds(
                            *bbox,
                            transform=vrt.transform
                        )

                        data = vrt.read(
                            window=window,
                            out_dtype=src.dtypes[0]
                        )

                        if data.size == 0:
                            print(f"    ⚠ No overlap → {raw_file.name}")
                            continue

                        out_meta = vrt.meta.copy()
                        out_meta.update({
                            "height": data.shape[1],
                            "width": data.shape[2],
                            "transform": vrt.window_transform(window),
                            "driver": "GTiff",
                            "compress": "deflate",
                            "tiled": True
                        })

                out_file = OUTPUT_DIR / raw_file.name

                with rasterio.open(out_file, "w", **out_meta) as dst:
                    dst.write(data)

                print(f"    ✔ {out_file.name}")

            except Exception as e:
                print(f"    ❌ Failed {raw_file.name}: {e}")

        # batch_tmp auto-deleted here

print("\n✅ All batches completed")


🚀 Batch 1/105


QUEUEING TASKS | :   0%|          | 0/25 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/25 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/25 [00:00<?, ?it/s]

  Downloaded 25 files
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160101.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160102.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160103.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160104.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160105.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160106.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160107.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160109.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160110.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160111.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160112.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160113.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160114.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160115.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160117.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160118.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160119.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20160121.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_201

...

  Downloaded 25 files
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240505.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240506.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240507.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240509.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240510.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240511.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240512.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240513.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240514.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240515.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240516.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240517.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240518.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240519.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240520.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240521.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240522.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240523.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_202

QUEUEING TASKS | :   0%|          | 0/23 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/23 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/23 [00:00<?, ?it/s]

  Downloaded 23 files
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240602.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240603.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240604.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240606.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240607.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240608.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240610.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240611.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240613.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240615.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240616.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240617.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240618.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240619.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240620.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240621.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240622.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20240623.tif
    ✔ NSIDC-0779_EASE2_G1km_SMAP_SM_DS_202

In [ ]:
import shutil
from google.colab import files

# Define the directory to be archived (using OUTPUT_DIR from previous cells)
folder_to_archive = OUTPUT_DIR # This variable is defined in a previous cell
archive_name = "NSIDC_0779_2016_25"

# Create a .zip archive of the folder
# The base_name argument is the name of the archive, without the .zip extension
# The format argument specifies the archive format (e.g., 'zip', 'tar', 'gztar', 'bztar', 'xztar')
# The root_dir argument is the directory to start archiving from
# The base_dir argument specifies the directory to archive from within root_dir
shutil.make_archive(base_name=archive_name, format='zip', root_dir=folder_to_archive.parent, base_dir=folder_to_archive.name)

print(f"Successfully created {archive_name}.zip")

# Automatically download the created zip file
files.download(f'{archive_name}.zip')

print("Download initiated.")

Successfully created NSIDC_0779_2016_25.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated.


In [ ]:
files = earthaccess.download(
    granules,
    local_path=DOWNLOAD_DIR
)


QUEUEING TASKS | :   0%|          | 0/223 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/223 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/223 [00:00<?, ?it/s]

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from shapely.geometry import box, mapping
from pathlib import Path
import os
import shutil
import tempfile

In [ ]:
import shutil
import os

# Define the directory to delete
# For demonstration, let's create a dummy directory first
dummy_dir = "/content/NSIDC_0779"

# Delete the directory and its contents
shutil.rmtree(dummy_dir)

if not os.path.exists(dummy_dir):
    print(f"Directory '{dummy_dir}' and its contents have been successfully deleted.")
else:
    print(f"Failed to delete directory '{dummy_dir}'.")

Directory '/content/NSIDC_0779' and its contents have been successfully deleted.


In [ ]:
granules[0]

Collection: {'EntryTitle': 'SMAP-Derived 1-km Downscaled Surface Soil Moisture Product V001'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 86.0, 'SouthBoundingCoordinate': -86.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2015-04-02T00:00:00.000Z', 'EndingDateTime': '2015-04-02T23:59:59.000Z'}}
Size(MB): 439.104
Data: ['https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/SMAP-Related/NSIDC-0779/1/2015/04/02/NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20150402.tif']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
collections = earthaccess.search_datasets(short_name="NSIDC-0779")
collections[0]

{
  "meta": {
    "revision-id": 27,
    "deleted": false,
    "format": "application/iso19115+xml",
    "provider-id": "NSIDC_CPRD",
    "has-combine": false,
    "user-id": "crumlyd",
    "has-formats": false,
    "associations": {
      "services": [
        "S2472217299-NSIDC_CPRD",
        "S3107386427-NSIDC_CPRD"
      ],
      "tools": [
        "TL1952642907-NSIDC_ECS",
        "TL1977971361-NSIDC_ECS",
        "TL3010466318-NSIDC_CPRD"
      ]
    },
    "s3-links": [
      "nsidc-cumulus-prod-protected/SMAP-Related/NSIDC-0779/1",
      "nsidc-cumulus-prod-public/SMAP-Related/NSIDC-0779/1"
    ],
    "has-spatial-subsetting": false,
    "native-id": "SMAP-Derived 1-km Downscaled Surface Soil Moisture Product V001",
    "has-transforms": false,
    "association-details": {
      "services": [
        {
          "concept-id": "S2472217299-NSIDC_CPRD"
        },
        {
          "concept-id": "S3107386427-NSIDC_CPRD"
        }
      ],
      "tools": [
        {
          "co

# **MODIS LST : MYD11A1**

In [ ]:
import ee
ee.Authenticate()

In [ ]:


roi = ee.Geometry.Rectangle([
    -122.62822473238745,
     34.910145527064195,
    -118.58047102622170,
     40.68211245220932
])

date = "2020-07-15"

collection = (
    ee.ImageCollection("MODIS/061/MYD11A1")
    .filterDate(date, ee.Date(date).advance(1, "day"))
    .filterBounds(roi)
)

def apply_qc(img):
    lst = img.select("LST_Day_1km")
    qc  = img.select("QC_Day")

    good = qc.bitwiseAnd(3).eq(0)   # bits 0–1
    lst = lst.updateMask(good)

    lst = lst.multiply(0.02)        # scale factor
    return lst.copyProperties(img, ["system:time_start"])


lst_img = (
    collection
    .map(apply_qc)
    .mosaic()
    .clip(roi)
)

import geemap

Map = geemap.Map(center=[37.5, -120.5], zoom=6)
Map.addLayer(
    lst_img,
    {"min": 270, "max": 320, "palette": ["blue", "green", "yellow", "red"]},
    "LST Day (K)"
)
Map



In [ ]:
from datetime import datetime, timedelta

start_date = datetime(2015,1,1)
end_date   = datetime(2025,1,1)

delta = timedelta(days=1)

d = start_date
while d <= end_date:
    date_str = d.strftime("%Y-%m-%d")

    collection = (
        ee.ImageCollection("MODIS/061/MYD11A1")
        .filterDate(date_str, ee.Date(date_str).advance(1,"day"))
        .filterBounds(roi)
    )

    lst_img = collection.map(apply_qc).mosaic().clip(roi)

    out_file = f"/content/MYD11A1/MYD11A1_LST_{date_str}.tif"
    geemap.ee_export_image(lst_img, filename=out_file, scale=1000, crs="EPSG:4326")

    print("✅ Downloaded:", out_file)
    d += delta


Streaming output truncated to the last 5000 lines.
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-21.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-21.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-22.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-22.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-23.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-23.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-24.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-24.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-25.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-25.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2019-09-26.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2019-09-26.tif
Generating URL ...
Please wait .

Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-20.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-20.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-21.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-21.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-22.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-22.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-23.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-23.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-24.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-24.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-25.tif
✅ Downloaded: /content/MYD11A1/MYD11A1_LST_2022-06-25.tif
Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1/MYD11A1_LST_2022-06-26.tif
✅

In [ ]:
import shutil
from google.colab import files

# Define the directory to be archived (using OUTPUT_DIR from previous cells)
folder_to_archive = "/content/MYD11A1" # This variable is defined in a previous cell
archive_name = "MYD11A1_2015_25"

# Create a .zip archive of the folder
# The base_name argument is the name of the archive, without the .zip extension
# The format argument specifies the archive format (e.g., 'zip', 'tar', 'gztar', 'bztar', 'xztar')
# The root_dir argument is the directory to start archiving from
# The base_dir argument specifies the directory to archive from within root_dir
shutil.make_archive(base_name=archive_name, format='zip', root_dir=folder_to_archive)

print(f"Successfully created {archive_name}.zip")

# Automatically download the created zip file
files.download(f'{archive_name}.zip')

print("Download initiated.")

Successfully created MYD11A1_2015_25.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated.


In [ ]:
# ----------------------------
# Export directly to Colab workspace
# ----------------------------
out_file = "/content/MYD11A1_LST_TEST.tif"

geemap.ee_export_image(
    lst_img,
    filename=out_file,
    scale=1000,         # 1 km resolution
    crs="EPSG:4326",
    file_per_band=False
)

print("✅ Downloaded to Colab:", out_file)

Generating URL ...
Please wait ...
Data downloaded to /content/MYD11A1_LST_TEST.tif
✅ Downloaded to Colab: /content/MYD11A1_LST_TEST.tif
